In [ ]:
import warnings, itertools, datetime
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, root_mean_squared_error, confusion_matrix

warnings.filterwarnings('ignore')
line = '\n'+'='*50+'\n'

In [ ]:
class ModelPipeline:
    """Classe qui encapsule les étapes de la preparation, l'entrainement et l'evaluation
       On peut créer des instances différentes pour chaque exercice.
       
       param task: regression ou classification
    """
    def __init__(self, preprocessing=None, model=None, data=None, target_name=None, task=None, skip_model=False):
        self.preprocess = preprocessing
        self.model = model
        self.df = self.preprocess(data) if self.preprocess else data
        self.target_name = target_name
        self.task = task
        self.skip_model = skip_model
        self.__check_init()
        self.X_train, self.X_test, self.y_train, self.y_test, self.y_pred = None, None, None, None, None
        self.is_trained = False
        self.score = 0

    def __check_init(self):
        if (self.model is None) and (not self.skip_model):
            raise Exception("Aucun modèle defini !")
        if not isinstance(self.df, pd.DataFrame):
            raise Exception("Aucun dataset defini !")
        if not self.target_name:
            raise Exception("Aucune variable cible definie !")
        
    def data_split(self, verbose, test_size=0.2):
        X = self.df[[c for c in self.df.columns if c != self.target_name]]
        y = self.df[self.target_name]
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=test_size, random_state=0)
        if verbose:
            print(
                f"Train X : {self.X_train.shape}",
                f"Test X : {self.X_test.shape}",
                f"train y : {self.y_train.shape}", 
                f"test y : {self.y_test.shape}",
                sep='\n' 
            )
        return self

    def train(self, verbose):
        start = datetime.datetime.now()
        self.model.fit(self.X_train, self.y_train)
        duree = datetime.datetime.now() - start
        if verbose:
            print(f"L'entrainement a duré {duree} !")
        self.is_trained = True
        return self

    def predict(self, new_df=None, verbose=True, append_to_new_df=False):
        assert self.is_trained, f"Le modèle doit etre d'abord entrainé"
        if new_df is None:
            if verbose:
                print('pas de dataset fourni, on prédit sur le set de test')
            new_df = self.X_test
        if verbose:
            print('Prédiction en cours')
        y_pred = self.model.predict(new_df)
        if append_to_new_df:
            new_df[self.target_name] = y_pred
            return new_df
        return y_pred
    
    def eval(self, new_df=None, verbose=True):
        self.y_pred = self.predict(new_df, verbose=verbose)
        if self.task == "regression":
            self.score = {'rmse': round(root_mean_squared_error(self.y_test, self.y_pred), 3)}
        elif self.task == "classification":
            self.score = {'accuracy': round(accuracy_score(self.y_test, self.y_pred), 3)}
        else:
            raise Exception(f"Task {self.task} doesnot exist. Choose between [classification, regression]")
        # round
        if verbose:
            print(f"Score eval : {self.score}")

    def train_and_eval(self, new_data=None, verbose=True):
        self.data_split(verbose=verbose).train(verbose=verbose).eval(new_data, verbose=verbose)
    
    def get_eval_score(self):
        return self.score

*Exemple d'un autre cours à adapter*

In [ ]:
# Dans le modèle, on va ajouter une étape qui permet de pre traiter les données en faisant la normalisation
scaler = ColumnTransformer(
    transformers=[
        ('minmax_scaler', MinMaxScaler(), ['hauteur_m', 'circonference_cm'])
    ], remainder='passthrough'
)

# ce pretraitement avec le scaler minmax est ajouté dans un pipeline qu'on passera à la classe ModelPipeline que j'ai utilisé dans le tp précédent
pipeline_LR = Pipeline([
    ('preprocessing', scaler),
    ('logistic_regression', LogisticRegression())
])

# et puisque la classe ModelPipeline encapsule déjà les étapes de train test split et de train et predict, il suffit de faire une instance et d'appeler ces méthodes
p_ = ModelPipeline(
    model = pipeline_LR,
    data = df_platane_stade_known, # on lui passe les données de l'espece platane avec les stades de develeoppement connus
    target_name = 'stade_de_developpement',
    task = 'classification'
)
# p_.data_split(verbose=True).train(verbose=True)
p_.train_and_eval()
p_.model

In [ ]:
# la sortie précédente montre que le modèle de regression logistique a prédit avec une précision de 75.6% sur le test set issu des données connues

# on suppose que cette précision nous suffit et on va utiliser ce modèle fitté pour prédire les stades de developpement des platanes non connus
res = p_.predict(new_df=df_platane_stade_nc, append_to_new_df=True)
res 

*et avec mlflow*

In [19]:
import os, tempfile, datetime
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, root_mean_squared_error

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer

import mlflow
import mlflow.sklearn


class ModelPipelineWithMlflow:
    """
    Classe qui encapsule les étapes de la préparation, l'entraînement et l'évaluation.
    Trace les expériences via MLflow et sauvegarde les artefacts dans un dossier temporaire.

    param task: 'regression' ou 'classification'
    """
    def __init__(self, preprocessing=None, model=None, data=None, target_name=None, task=None, skip_model=False, experiment_name="Default"):
        self.preprocess = preprocessing
        self.model = model
        self.df = self.preprocess(data) if self.preprocess else data
        self.target_name = target_name
        self.task = task
        self.skip_model = skip_model
        self.__check_init()
        self.X_train, self.X_test, self.y_train, self.y_test, self.y_pred = None, None, None, None, None
        self.is_trained = False
        self.score = 0
        self.experiment_name = experiment_name
        self.artifact_dir = "tmp" # tempfile.mkdtemp()  # Dossier temporaire pour sauvegarder les artefacts

        # Configurer l'expérience MLflow
        mlflow.set_experiment(self.experiment_name)

    def __check_init(self):
        if (self.model is None) and (not self.skip_model):
            raise Exception("Aucun modèle défini !")
        if not isinstance(self.df, pd.DataFrame):
            raise Exception("Aucun dataset défini !")
        if not self.target_name:
            raise Exception("Aucune variable cible définie !")

    def data_split(self, verbose=True, test_size=0.2):
        X = self.df[[c for c in self.df.columns if c != self.target_name]]
        y = self.df[self.target_name]
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=test_size, random_state=0)
        if verbose:
            print(
                f"Train X : {self.X_train.shape}",
                f"Test X : {self.X_test.shape}",
                f"Train y : {self.y_train.shape}",
                f"Test y : {self.y_test.shape}",
                sep='\n'
            )
        mlflow.log_param("train_shape", self.X_train.shape)
        mlflow.log_param("test_size", test_size)
        mlflow.log_param("num_features", X.shape[1])
        mlflow.log_param("num_samples", X.shape[0])
        return self

    def train(self, verbose=True):
        start = datetime.datetime.now()
        self.model.fit(self.X_train, self.y_train)
        duration = datetime.datetime.now() - start
        if verbose:
            print(f"L'entraînement a duré {duration} !")
        mlflow.log_metric("training_duration", duration.total_seconds())
        self.is_trained = True

        # Sauvegarder le modèle entraîné
        model_path = os.path.join(self.artifact_dir, "model.pkl")
        mlflow.sklearn.save_model(self.model, model_path)
        mlflow.log_artifact(model_path)
        return self

    def predict(self, new_df=None, verbose=True, append_to_new_df=False):
        assert self.is_trained, "Le modèle doit être entraîné d'abord."
        if new_df is None:
            if verbose:
                print("Pas de dataset fourni, on prédit sur le set de test.")
            new_df = self.X_test
        if verbose:
            print("Prédiction en cours...")
        y_pred = self.model.predict(new_df)
        if append_to_new_df:
            new_df[self.target_name] = y_pred
            return new_df
        return y_pred

    def eval(self, new_df=None, verbose=True):
        self.y_pred = self.predict(new_df, verbose=verbose)
        if self.task == "regression":
            rmse = root_mean_squared_error(self.y_test, self.y_pred)
            self.score = {'rmse': round(rmse, 3)}
            mlflow.log_metric("rmse", rmse)
        elif self.task == "classification":
            accuracy = accuracy_score(self.y_test, self.y_pred)
            self.score = {'accuracy': round(accuracy, 3)}
            mlflow.log_metric("accuracy", accuracy)
        else:
            raise Exception(f"Task {self.task} n'existe pas. Choisissez entre 'classification' ou 'regression'.")

        if verbose:
            print(f"Score d'évaluation : {self.score}")
        return self.score

    def train_and_eval(self, new_data=None, verbose=True):
        """
        Enchaîne les étapes de split, entraînement et évaluation.
        """
        with mlflow.start_run():
            self.data_split(verbose=verbose).train(verbose=verbose).eval(new_data, verbose=verbose)
            mlflow.log_params({
                "model": self.model.__class__.__name__,
                "task": self.task
            })

    def get_eval_score(self):
        return self.score


In [20]:
def load_iris_dataset():
    iris = load_iris()
    data = pd.DataFrame(iris.data, columns=iris.feature_names)
    data["target"] = iris.target
    return data

iris_data = load_iris_dataset()

features_names = [c for c in iris_data.columns if c != 'target']

scaler = ColumnTransformer(
    transformers=[
        ('standard_scaler', StandardScaler(), features_names)
    ], remainder='passthrough'
)

model = Pipeline([
    ('preprocessing', scaler),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
    
pipeline = ModelPipelineWithMlflow(
    model=model,
    data=iris_data,
    target_name="target",
    task="classification",
    experiment_name="iris test"
)

pipeline.train_and_eval(verbose=True)
# mlflow ui in cli to luach

2025/01/04 14:59:15 INFO mlflow.tracking.fluent: Experiment with name 'iris test' does not exist. Creating a new experiment.


Train X : (120, 4)
Test X : (30, 4)
Train y : (120,)
Test y : (30,)
L'entraînement a duré 0:00:00.745984 !
Pas de dataset fourni, on prédit sur le set de test.
Prédiction en cours...
Score d'évaluation : {'accuracy': 1.0}
🏃 View run gregarious-finch-421 at: http://127.0.0.1:5000/#/experiments/149456351687505036/runs/1808f25dc2724d45b4ae4c4b95fdc0a9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/149456351687505036


In [21]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000") 
client = MlflowClient()

experiments = client.search_experiments() 
experiment_names = [exp.name for exp in experiments]
print(experiment_names)

['iris test', 'Default']


In [26]:
#### AUTRES TEST

import os
import datetime
import tempfile
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.pipeline import Pipeline
from joblib import dump, load


def root_mean_squared_error(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)


class ModelPipelineWithMlflow:
    """
    Classe qui encapsule les étapes de préparation, d'entraînement et d'évaluation.
    Trace les expériences via MLflow et sauvegarde les artefacts dans un dossier temporaire.
    
    Si le modèle est un pipeline, les étapes de preprocessing sont sauvegardées séparément.
    
    param task: 'regression' ou 'classification'
    """

    def __init__(self, preprocessing=None, model=None, data=None, target_name=None, task=None, skip_model=False, experiment_name="Default"):
        self.preprocess = preprocessing
        self.model = model
        self.df = self.preprocess(data) if self.preprocess else data
        self.target_name = target_name
        self.task = task
        self.skip_model = skip_model
        self.__check_init()
        self.X_train, self.X_test, self.y_train, self.y_test, self.y_pred = None, None, None, None, None
        self.is_trained = False
        self.score = 0
        self.experiment_name = experiment_name
        self.artifact_dir = "tmp"

        # Configurer l'expérience MLflow
        mlflow.set_experiment(self.experiment_name)

    def __check_init(self):
        if (self.model is None) and (not self.skip_model):
            raise Exception("Aucun modèle défini !")
        if not isinstance(self.df, pd.DataFrame):
            raise Exception("Aucun dataset défini !")
        if not self.target_name:
            raise Exception("Aucune variable cible définie !")

    def data_split(self, verbose=True, test_size=0.2):
        X = self.df[[c for c in self.df.columns if c != self.target_name]]
        y = self.df[self.target_name]
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=test_size, random_state=0)
        if verbose:
            print(
                f"Train X : {self.X_train.shape}",
                f"Test X : {self.X_test.shape}",
                f"Train y : {self.y_train.shape}",
                f"Test y : {self.y_test.shape}",
                sep='\n'
            )
        mlflow.log_param("test_size", test_size)
        mlflow.log_param("num_features", X.shape[1])
        mlflow.log_param("num_samples", X.shape[0])
        return self

    def save_pipeline_steps(self):
        """
        Si le modèle est un pipeline, sauvegarde chaque étape séparément.
        """
        if isinstance(self.model, Pipeline):
            steps = self.model.named_steps
            for name, step in steps.items():
                step_path = os.path.join(self.artifact_dir, f"{name}_.pkl")
                dump(step, step_path)
                mlflow.log_artifact(step_path, artifact_path=f"pipeline_steps/{name}")

    def load_pipeline_steps(self):
        """
        Recharge les étapes du pipeline depuis les artefacts sauvegardés.
        """
        if isinstance(self.model, Pipeline):
            steps = {}
            for name, step in self.model.named_steps.items():
                step_path = os.path.join(self.artifact_dir, f"{name}_.pkl")
                steps[name] = load(step_path)
            self.model = Pipeline([(name, steps[name]) for name in steps])

    def train(self, verbose=True):
        start = datetime.datetime.now()

        # Si c'est un pipeline, fit les encodeurs séparément
        if isinstance(self.model, Pipeline):
            self.save_pipeline_steps()
            preprocessors = self.model.named_steps
            preprocessors["model"].fit(self.X_train, self.y_train)
        else:
            self.model.fit(self.X_train, self.y_train)

        duration = datetime.datetime.now() - start
        if verbose:
            print(f"L'entraînement a duré {duration} !")
        mlflow.log_metric("training_duration", duration.total_seconds())
        self.is_trained = True

        # Sauvegarder le modèle entraîné
        model_path = os.path.join(self.artifact_dir, "model.pkl")
        mlflow.sklearn.save_model(self.model, model_path)
        mlflow.log_artifact(model_path)
        return self

    def predict(self, new_df=None, verbose=True, append_to_new_df=False):
        assert self.is_trained, "Le modèle doit être entraîné d'abord."
        if new_df is None:
            if verbose:
                print("Pas de dataset fourni, on prédit sur le set de test.")
            new_df = self.X_test
        if verbose:
            print("Prédiction en cours...")
        y_pred = self.model.predict(new_df)
        if append_to_new_df:
            new_df[self.target_name] = y_pred
            return new_df
        return y_pred

    def eval(self, new_df=None, verbose=True):
        self.y_pred = self.predict(new_df, verbose=verbose)
        if self.task == "regression":
            rmse = root_mean_squared_error(self.y_test, self.y_pred)
            self.score = {'rmse': round(rmse, 3)}
            mlflow.log_metric("rmse", rmse)
        elif self.task == "classification":
            accuracy = accuracy_score(self.y_test, self.y_pred)
            self.score = {'accuracy': round(accuracy, 3)}
            mlflow.log_metric("accuracy", accuracy)
        else:
            raise Exception(f"Task {self.task} n'existe pas. Choisissez entre 'classification' ou 'regression'.")

        if verbose:
            print(f"Score d'évaluation : {self.score}")
        return self.score

    def train_and_eval(self, new_data=None, verbose=True):
        """
        Enchaîne les étapes de split, entraînement et évaluation.
        """
        with mlflow.start_run():
            self.data_split(verbose=verbose).train(verbose=verbose).eval(new_data, verbose=verbose)
            mlflow.log_params({
                "model": self.model.__class__.__name__,
                "task": self.task
            })

    def get_eval_score(self):
        return self.score
    


def load_iris_dataset():
    iris = load_iris()
    data = pd.DataFrame(iris.data, columns=iris.feature_names)
    data["target"] = iris.target
    return data

iris_data = load_iris_dataset()

features_names = [c for c in iris_data.columns if c != 'target']

scaler = ColumnTransformer(
    transformers=[
        ('standard_scaler', StandardScaler(), features_names)
    ], remainder='passthrough'
)

model = Pipeline([
    ('preprocessing', scaler),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])
    
pipeline = ModelPipelineWithMlflow(
    model=model,
    data=iris_data,
    target_name="target",
    task="classification",
    experiment_name="iris test"
)

pipeline.train_and_eval(verbose=True)
# mlflow ui in cli to luach


Train X : (120, 4)
Test X : (30, 4)
Train y : (120,)
Test y : (30,)
L'entraînement a duré 0:00:01.093149 !
🏃 View run unique-asp-921 at: http://127.0.0.1:5000/#/experiments/149456351687505036/runs/a2aa1ed381854b8aaa6d0d6adc572f9b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/149456351687505036


MlflowException: Path 'tmp\model.pkl' already exists and is not empty